## Abstract ##

This notebook observes the popular puzzle of Sudoku through the lens of set theory and predicate logic. Although the puzzle is defined by three simple constraints, logic and constraint have led to the development of a number of effective strategies, which are essential for cracking more advanced puzzles and obtaining the unique solution. We will take a look at the strategies required to solve most of the solvable puzzles and apply them to a selected sample varying in difficulty. Finally, we will observe a 3D interpretation of the solution path.

## Sudoku basics ##
A Sudoku puzzle is a partially filled $9 \times 9$ grid, for which each row, column and $3 \times 3$ box contains the digits from $1$ to $9$ exactly once. A puzzle in its initial, partially filled state has one unique solution, meaning that the digits can be distributed on the board in one particular way only. Gary McGuire and collaborators proved that no proper Sudoku puzzle with a unique solution can have fewer than 17 clues, settling the long-standing minimum-clue problem in 2012. However, a unique solution does not necessarily mean that a puzzle can be solved by logical deduction alone. Some of the hardest puzzles require backtracking, and we will not be considering those in our algorithm testing.
<br>A Sudoku puzzle can be viewed in terms of candidate sets. For every empty cell, one can consider the set of digits that do not violate the row, column, and box conditions. In this way, a Sudoku puzzle may be viewed not only as a $9 \times 9$ grid of digits, but also as a $9 \times 9$ grid of candidate sets. For a solved or initially given cell, the candidate set consists of the single assigned digit, whereas for an empty cell it contains all values that are still possible. Solving Sudoku then becomes a process of progressively reducing these candidate sets until forced placements can be made. In particular, if a candidate set has size $1$, then the corresponding digit must be placed in that cell.



## Formalizing Sudoku as Sets and Predicate Logic
We can start by defying the set of digits that may appear in a puzzle as
$$ D = \{1,2,...,9\}. $$
A standard sudoku grid of $9 \times 9$ contains $81$ cells, each of them identified by a pair of coordinates $(i,j)$, where i denotes the row index and j the column index. The set of all cells is therefore defined by
$$G = \{(i,j) \ | \ 1\leq i \leq 9,\  1\leq j \leq 9\}.$$
A complete Sudku solution may be viewed as a relation $R \subseteq G \times D$, where $G \times D$ is the Cartesian product of the set of cells $G$ and set of digits $D$. In this formulation, $R$ consists of ordered pairs of the form $((i,j),d)$, where $(i,j) \in G$ and $d \in D$. The interpretaton of $((i,j),d) \in R$ is that the cell $(i,j)$ contains the digit $d$.
<br> Equivalently, one may define a predicate $P(i,j,d)$ by
$$P(i,j,d) \ \mbox{is true if and only if} \ ((i,j),d) \in R.$$
Thus, the predicate $P(i,j,d)$ expresses the statement that digit $d$ is placed in cell $(i,j)$.

## Strategy definition ##

## Code ##

In [50]:
from itertools import combinations
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
%matplotlib notebook

In [51]:
sdk1 = '084520000000007000306908000200600800065000740008005002000106508000200000000089230'
sdk = '009603004010700080400080003000000641005000200164000000600010002050002060300407500'
def string_to_board(s):
    board = []
    for i in range(0,81,9):
        row = [int(x) for x in s[i:i+9]]
        board.append(row)
    return board


board = string_to_board(sdk)


In [52]:
# Board manual input
#
# board = [
#     [0,0,0,0,0,0,0,0,0],
#     [0,0,0,0,0,0,0,0,0],
#     [0,0,0,0,0,0,0,0,0],
#     [0,0,0,0,0,0,0,0,0],
#     [0,0,0,0,0,0,0,0,0],
#     [0,0,0,0,0,0,0,0,0],
#     [0,0,0,0,0,0,0,0,0],
#     [0,0,0,0,0,0,0,0,0],
#     [0,0,0,0,0,0,0,0,0],
# ]
#
# draw_board(board)

In [53]:
def draw_board(board):
    for r in range(9):
        if r % 3 == 0 and r != 0:
            print("-"*21)

        row = ""
        for c in range(9):
            if c % 3 == 0 and c != 0:
                row += "| "

            val = board[r][c] if board[r][c] != 0 else "."
            row += str(val) + " "

        print(row)
draw_board(board)

. . 9 | 6 . 3 | . . 4 
. 1 . | 7 . . | . 8 . 
4 . . | . 8 . | . . 3 
---------------------
. . . | . . . | 6 4 1 
. . 5 | . . . | 2 . . 
1 6 4 | . . . | . . . 
---------------------
6 . . | . 1 . | . . 2 
. 5 . | . . 2 | . 6 . 
3 . . | 4 . 7 | 5 . . 


In [54]:
# Candidates functions

def get_candidates(board, row, col):
    if board[row][col] != 0:
        return set()

    digits = set(range(1, 10))

    row_nums = set(board[row])

    col_nums = {board[r][col] for r in range(9)}

    start_row = (row // 3) * 3
    start_col = (col // 3) * 3

    box_nums = {
        board[r][c]
        for r in range(start_row, start_row + 3)
        for c in range(start_col, start_col + 3)
    }

    used = row_nums | col_nums | box_nums
    used.discard(0)

    return digits - used

def get_all_candidates(board):
    candidates = [[set() for _ in range(9)] for _ in range(9)]

    for row in range(9):
        for col in range(9):
            if board[row][col] == 0:
                candidates[row][col] = get_candidates(board, row, col)

    return candidates


In [55]:
# Naked Singles

def naked_singles(board, candidates):
    progress = False

    for row in range(9):
        for col in range(9):
            if board[row][col] == 0 and len(candidates[row][col]) == 1:
                value = next(iter(candidates[row][col]))
                board[row][col] = value
                print(f'Naked single: placed {value} at cell ({row+1},{col+1})')
                progress = True

    return progress

In [56]:
# Hidden Singles

def hidden_singles_rows(board, candidates):

    for row in range(9):
        digit_positions = {digit: [] for digit in range(1, 10)}

        for col in range(9):
            if board[row][col] == 0:
                for digit in candidates[row][col]:
                    digit_positions[digit].append((row, col))

        for digit in range(1,10):
            if len(digit_positions[digit]) == 1:
                r, c = digit_positions[digit][0]
                board[r][c] = digit
                print(f'Hidden single (row): placed {digit} placed at cell ({r+1},{c+1})')
                return True
    return False

def hidden_singles_columns(board, candidates):

    for col in range(9):
        digit_positions = {digit: [] for digit in range(1, 10)}

        for row in range(9):
            if board[row][col] == 0:
                for digit in candidates[row][col]:
                    digit_positions[digit].append((row, col))

        for digit in range(1,10):
            if len(digit_positions[digit]) == 1:
                r, c = digit_positions[digit][0]
                board[r][c] = digit
                print(f'Hidden single (column): placed {digit} placed at cell ({r+1},{c+1})')
                return True
    return False

def hidden_singles_boxes(board, candidates):

    for box_row_start in range(0,9,3):
        for box_col_start in range(0,9,3):
            digit_positions = {digit: [] for digit in range(1, 10)}

            for row in range(box_row_start,box_row_start+3):
                for col in range(box_col_start,box_col_start+3):
                    if board[row][col] == 0:
                        for digit in candidates[row][col]:
                            digit_positions[digit].append((row, col))

            for digit in range(1,10):
                if len(digit_positions[digit]) == 1:
                    r, c = digit_positions[digit][0]
                    board[r][c] = digit
                    print(f'Hidden single (box): placed {digit} placed at cell ({r+1},{c+1})')
                    return True
    return False

In [57]:
# Helper Functions

def get_row_cells(row):
    return [(row, col) for col in range(9)]

def get_col_cells(col):
    return [(row, col) for row in range(9)]

def get_box_cells(start_row, start_col):
    return [
        (row, col)
        for row in range(start_row, start_row + 3)
        for col in range(start_col, start_col + 3)
    ]

In [58]:
# Naked Pairs

def naked_sets_in_unit(board, candidates, unit_cells, set_size):
    progress = False
    if set_size == 2:               # Helper variable used in the print message
        print_variable = 'pairs'
    elif set_size == 3:
        print_variable = 'triples'


    candidate_cells = [
        (row, col)
        for row, col in unit_cells
        if board[row][col] == 0 and 2 <= len(candidates[row][col]) <= set_size
    ]

    for cell_combo in combinations(candidate_cells, set_size):
        combo_digits = set()

        for row, col in cell_combo:
            combo_digits.update(candidates[row][col])

        if len(combo_digits) == set_size:
            combo_cells = set(cell_combo)

            for row, col in unit_cells:
                if (row, col) not in combo_cells and board[row][col] == 0:
                    before = candidates[row][col].copy()
                    candidates[row][col] -= combo_digits

                    if candidates[row][col] != before:
                        cell_combo_print = []    # Helper variable to visualize cells as numbers 1-9 and not indices
                        for x in cell_combo:
                            cell_combo_print.append((x[0]+1, x[1]+1))

                        print(
                            f'Naked {print_variable}: Removed digits {tuple(combo_digits)} from cell {(row+1, col+1)} '
                            f'using cells {tuple(cell_combo_print)}'
                        )
                        progress = True

            if progress:
                return True

    return False

def naked_pairs_rows(board, candidates):
    progress = False

    for row in range(9):
        if naked_sets_in_unit(board, candidates, get_row_cells(row), 2):
            progress = True

    return progress

def naked_pairs_cols(board, candidates):
    progress = False

    for col in range(9):
        if naked_sets_in_unit(board, candidates, get_col_cells(col), 2):
            progress = True

    return progress

def naked_pairs_boxes(board, candidates):
    progress = False

    for start_row in range(0, 9, 3):
        for start_col in range(0, 9, 3):
            if naked_sets_in_unit(board, candidates, get_box_cells(start_row, start_col), 2):
                progress = True

    return progress

def naked_pairs(board, candidates):
    if naked_pairs_rows(board, candidates):
        return True
    if naked_pairs_cols(board, candidates):
        return True
    if naked_pairs_boxes(board, candidates):
        return True
    return False

def naked_triples_rows(board, candidates):
    progress = False

    for row in range(9):
        if naked_sets_in_unit(board, candidates, get_row_cells(row), 3):
            progress = True

    return progress

def naked_triples_cols(board, candidates):
    progress = False

    for col in range(9):
        if naked_sets_in_unit(board, candidates, get_col_cells(col), 3):
            progress = True

    return progress

def naked_triples_boxes(board, candidates):
    progress = False

    for start_row in range(0, 9, 3):
        for start_col in range(0, 9, 3):
            if naked_sets_in_unit(board, candidates, get_box_cells(start_row, start_col), 3):
                progress = True

    return progress

def naked_triples(board, candidates):
    if naked_triples_rows(board, candidates):
        return True
    if naked_triples_cols(board, candidates):
        return True
    if naked_triples_boxes(board, candidates):
        return True
    return False

In [59]:
# Hidden Pairs/Triples

def hidden_sets_in_unit(board, candidates, unit_cells, set_size):
    progress = False
    if set_size == 2:               # Helper variable used in the print message
        print_variable = 'pairs'
    elif set_size == 3:
        print_variable = 'triples'

    digit_positions = {digit: [] for digit in range(1,10)}

    for row, col in unit_cells:
        if board[row][col] == 0:
            for digit in candidates[row][col]:
                digit_positions[digit].append((row, col))

    candidate_digits = [
        digit for digit in range(1, 10)
        if 1 <= len(digit_positions[digit]) <= set_size
    ]

    for digit_combo in combinations(candidate_digits, set_size):
        combo_set = set(digit_combo)
        combo_cells = set()

        for digit in digit_combo:
            combo_cells.update(digit_positions[digit])

        if len(combo_cells) == set_size:
            for row, col in combo_cells:
                before = candidates[row][col].copy()
                candidates[row][col] &= combo_set

                if candidates[row][col] != before:
                    print(
                        f'Hidden {print_variable}: Remove digits {tuple(combo_set)} from cell {(row+1, col+1)}'
                        #f'from {before} to {candidates[row][col]}'
                    )
                    progress = True

            if progress:
                return True

    return False

def hidden_pairs_rows(board, candidates):
    progress = False

    for row in range(9):
        if hidden_sets_in_unit(board, candidates, get_row_cells(row), 2):
            progress = True

    return progress

def hidden_pairs_cols(board, candidates):
    progress = False

    for col in range(9):
        if hidden_sets_in_unit(board, candidates, get_col_cells(col), 2):
            progress = True

    return progress

def hidden_pairs_boxes(board, candidates):
    progress = False

    for start_row in range(0, 9, 3):
        for start_col in range(0, 9, 3):
            if hidden_sets_in_unit(board, candidates, get_box_cells(start_row, start_col), 2):
                progress = True

    return progress

def hidden_pairs(board, candidates):
    if hidden_pairs_rows(board, candidates):
        return True
    if hidden_pairs_cols(board, candidates):
        return True
    if hidden_pairs_boxes(board, candidates):
        return True
    return False

def hidden_triples_rows(board, candidates):
    progress = False

    for row in range(9):
        if hidden_sets_in_unit(board, candidates, get_row_cells(row), 3):
            progress = True

    return progress

def hidden_triples_cols(board, candidates):
    progress = False

    for col in range(9):
        if hidden_sets_in_unit(board, candidates, get_col_cells(col), 3):
            progress = True

    return progress

def hidden_triples_boxes(board, candidates):
    progress = False

    for start_row in range(0, 9, 3):
        for start_col in range(0, 9, 3):
            if hidden_sets_in_unit(board, candidates, get_box_cells(start_row, start_col), 3):
                progress = True

    return progress

def hidden_triples(board, candidates):
    if hidden_triples_rows(board, candidates):
        return True
    if hidden_triples_cols(board, candidates):
        return True
    if hidden_triples_boxes(board, candidates):
        return True

    return False

In [60]:
# Pointing Pairs/Triples

def pointing_pairs_triples(board, candidates):

    for r in range(0,9,3):
        for c in range(0,9,3):
            digit_positions = {digit: [] for digit in range(1,10)}

            for row in range(r, r+3):
                for col in range(c, c+3):
                    if board[row][col] == 0:
                        for digit in candidates[row][col]:
                            digit_positions[digit].append((row,col))

            for digit in range(1,10):
                cells = digit_positions[digit]

                # Rows
                if len(cells) == 2 and cells[0][0] == cells[1][0]:
                    target_row = cells[0][0]
                    #print(f'Pointing pair of {digit}s in ROW {target_row}')
                    for x in range(9):
                        if not (c <= x < c+3) and board[target_row][x] == 0:
                            before = candidates[target_row][x].copy()
                            candidates[target_row][x].discard(digit)

                            if candidates[target_row][x] != before:
                                print(f'Pointing pair: {digit}s removed from row {target_row+1}')
                                return True

                if len(cells) == 3 and cells[0][0] == cells[1][0] == cells[2][0]:
                    target_row = cells[0][0]
                    #print(f'Pointing triple of {digit}s in ROW {target_row}')
                    for x in range(9):
                        if not (c <= x < c+3) and board[target_row][x] == 0:
                            before = candidates[target_row][x].copy()
                            candidates[target_row][x].discard(digit)

                            if candidates[target_row][x] != before:
                                print(f'Pointing triple: {digit}s removed from row {target_row+1}')
                                return True

                # Columns
                if len(cells) == 2 and cells[0][1] == cells[1][1]:
                    target_col = cells[0][1]
                    #print(f'Pointing pair of {digit}s in COL {target_col}')
                    for x in range(9):
                        if not (r <= x < r+3) and board[x][target_col] == 0:
                            before = candidates[x][target_col].copy()
                            candidates[x][target_col].discard(digit)

                            if candidates[x][target_col] != before:
                                print(f'Pointing pair: {digit}s removed from COL {target_col+1}')
                                return True

                if len(cells) == 3 and cells[0][1] == cells[1][1] == cells[2][1]:
                    target_col = cells[0][1]
                    #print(f'Pointing triple of {digit}s in COL {target_col}')
                    for x in range(9):
                        if not (r <= x < r+3) and board[x][target_col] == 0:
                            before = candidates[x][target_col].copy()
                            candidates[x][target_col].discard(digit)

                            if candidates[x][target_col] != before:
                                print(f'Pointing triple: {digit}s removed from COL {target_col+1}')
                                return True

    return False

draw_board(board)

. . 9 | 6 . 3 | . . 4 
. 1 . | 7 . . | . 8 . 
4 . . | . 8 . | . . 3 
---------------------
. . . | . . . | 6 4 1 
. . 5 | . . . | 2 . . 
1 6 4 | . . . | . . . 
---------------------
6 . . | . 1 . | . . 2 
. 5 . | . . 2 | . 6 . 
3 . . | 4 . 7 | 5 . . 


In [61]:
# Box-Line Reduction

def box_line_reduction_rows(board, candidates):
    for r in range(9):
        digit_positions = {digit: [] for digit in range(1,10)}
        for c in range(9):
            if board[r][c] == 0:
                for digit in candidates[r][c]:
                    digit_positions[digit].append((r,c))

        for digit in range (1,10):
            cells = digit_positions[digit]

            if len(cells) in {2,3}:
                possible_cell_cols = {cells[x][1] for x in range(len(cells))}

                if possible_cell_cols.issubset({0,1,2}) or possible_cell_cols.issubset({3,4,5}) or possible_cell_cols.issubset({6,7,8}):
                    block_row_start = (r // 3) * 3
                    block_col_start = (cells[0][1] // 3) * 3
                    for row in range(block_row_start, block_row_start + 3):
                        for col in range(block_col_start, block_col_start + 3):
                            if row != r and board[row][col] == 0:
                                before = candidates[row][col].copy()
                                candidates[row][col].discard(digit)

                                if candidates[row][col] != before:
                                    print(f'Box/line reduction: {digit}s in ROW {r}, removed from BOX {block_row_start+1},{block_col_start+1} candidates')
                                    return True
    return False

def box_line_reduction_cols(board, candidates):
    for c in range(9):
        digit_positions = {digit: [] for digit in range(1,10)}
        for r in range(9):
            if board[r][c] == 0:
                for digit in candidates[r][c]:
                    digit_positions[digit].append((r,c))

        for digit in range (1,10):
            cells = digit_positions[digit]

            if len(cells) in {2,3}:
                possible_cell_rows = {cells[x][0] for x in range(len(cells))}

                if possible_cell_rows.issubset({0,1,2}) or possible_cell_rows.issubset({3,4,5}) or possible_cell_rows.issubset({6,7,8}):
                    block_row_start = (cells[0][0] // 3) * 3
                    block_col_start = (c // 3) * 3
                    for row in range(block_row_start, block_row_start + 3):
                        for col in range(block_col_start, block_col_start + 3):
                            if col != c and board[row][col] == 0:
                                before = candidates[row][col].copy()
                                candidates[row][col].discard(digit)

                                if candidates[row][col] != before:
                                    print(f'Box/line reduction for {digit}s in COL {c}, removed from BOX {block_row_start+1},{block_col_start+1} candidates')
                                    return True
    return False

def box_line_reduction(board, candidates):
    if box_line_reduction_rows(board, candidates):
        return True

    if box_line_reduction_cols(board, candidates):
        return True

    return False

draw_board(board)

. . 9 | 6 . 3 | . . 4 
. 1 . | 7 . . | . 8 . 
4 . . | . 8 . | . . 3 
---------------------
. . . | . . . | 6 4 1 
. . 5 | . . . | 2 . . 
1 6 4 | . . . | . . . 
---------------------
6 . . | . 1 . | . . 2 
. 5 . | . . 2 | . 6 . 
3 . . | 4 . 7 | 5 . . 


In [62]:
# X-Wing

def x_wing_rows(board, candidates):
    for digit in range(1, 10):
        row_positions = {}              # key is row number, value is (only) a pair of columns for digit

        for row in range(9):
            cols = []
            for col in range(9):
                if board[row][col] == 0 and digit in candidates[row][col]:
                    cols.append(col)

            if len(cols) == 2:
                row_positions[row] = cols

        rows = list(row_positions.keys())

        for i in range(len(rows)):
            for j in range(i + 1, len(rows)):
                r1, r2 = rows[i], rows[j]

                if row_positions[r1] == row_positions[r2]:
                    c1, c2 = row_positions[r1]

                    for row in range(9):
                        if row not in {r1, r2}:
                            for col in [c1, c2]:
                                if board[row][col] == 0:
                                    before = candidates[row][col].copy()
                                    candidates[row][col].discard(digit)

                                    if candidates[row][col] != before:
                                        print(f'X-Wing: removed {digit} from cell ({row+1}, {col+1}) using rows ({r1+1},{r2+1}) and cols ({c1+1},{c2+1})')
                                        return True

    return False

def x_wing_cols(board, candidates):
    for digit in range(1, 10):
        col_positions = {}

        for col in range(9):
            rows = []
            for row in range(9):
                if board[row][col] == 0 and digit in candidates[row][col]:
                    rows.append(row)

            if len(rows) == 2:
                col_positions[col] = rows

        cols = list(col_positions.keys())

        for i in range(len(cols)):
            for j in range(i + 1, len(cols)):
                c1, c2 = cols[i], cols[j]

                if col_positions[c1] == col_positions[c2]:
                    r1, r2 = col_positions[c1]

                    for col in range(9):
                        if col not in {c1, c2}:
                            for row in [r1, r2]:
                                if board[row][col] == 0:
                                    before = candidates[row][col].copy()
                                    candidates[row][col].discard(digit)

                                    if candidates[row][col] != before:
                                        print(f'X-Wing: removed {digit} from cell ({row+1}, {col+1}) using rows ({r1+1},{r2+1}) and cols ({c1+1},{c2+1})')
                                        return True

    return False

def x_wing(board, candidates):
    if x_wing_rows(board, candidates):
        return True
    if x_wing_cols(board, candidates):
        return True
    return False


In [63]:
# Y-Wing

def cells_see_each_other(cell1, cell2):
    r1, c1 = cell1
    r2, c2 = cell2

    return (
        r1 == r2 or
        c1 == c2 or
        (r1 // 3 == r2 // 3 and c1 // 3 == c2 // 3)
    )

def get_peers(row, col):
    peers = set()

    for c in range(9):
        if c != col:
            peers.add((row, c))

    for r in range(9):
        if r != row:
            peers.add((r, col))

    box_row_start = (row // 3) * 3
    box_col_start = (col // 3) * 3

    for r in range(box_row_start, box_row_start + 3):
        for c in range(box_col_start, box_col_start + 3):
            if (r, c) != (row, col):
                peers.add((r, c))

    return peers

def y_wing(board, candidates):
    bivalue_cells = []

    # collect all cells with exactly 2 candidates
    for row in range(9):
        for col in range(9):
            if board[row][col] == 0 and len(candidates[row][col]) == 2:
                bivalue_cells.append((row, col))

    # try each bivalue cell as the pivot
    for pivot in bivalue_cells:
        pr, pc = pivot
        pivot_set = candidates[pr][pc]

        # find first wing
        for wing1 in bivalue_cells:
            if wing1 == pivot:
                continue

            if not cells_see_each_other(pivot, wing1):
                continue

            w1r, w1c = wing1
            wing1_set = candidates[w1r][w1c]

            shared1 = pivot_set & wing1_set
            if len(shared1) != 1:
                continue

            # find second wing
            for wing2 in bivalue_cells:
                if wing2 == pivot or wing2 == wing1:
                    continue

                if not cells_see_each_other(pivot, wing2):
                    continue

                w2r, w2c = wing2
                wing2_set = candidates[w2r][w2c]

                shared2 = pivot_set & wing2_set
                if len(shared2) != 1:
                    continue

                # wings must share different pivot digits
                if shared1 == shared2:
                    continue

                # together, pivot + wings must use exactly 3 digits
                union_digits = pivot_set | wing1_set | wing2_set
                if len(union_digits) != 3:
                    continue

                # the digit shared by the two wings but not in the pivot is the elimination digit
                elim_digit_set = (wing1_set & wing2_set) - pivot_set
                if len(elim_digit_set) != 1:
                    continue

                elim_digit = next(iter(elim_digit_set))

                # eliminate from cells that see both wings
                common_peers = get_peers(w1r, w1c) & get_peers(w2r, w2c)

                for row, col in common_peers:
                    if board[row][col] == 0:
                        before = candidates[row][col].copy()
                        candidates[row][col].discard(elim_digit)

                        if candidates[row][col] != before:
                            pivot_print = tuple([x+1 for x in pivot])
                            wing1_print = tuple([x+1 for x in wing1])
                            wing2_print = tuple([z+1 for z in wing2])
                            print(
                                f'Y-Wing: removed {elim_digit} from cell ({row+1}, {col+1}) '
                                f'using pivot {pivot_print}, wing1 {wing1_print}, wing2 {wing2_print}'
                            )
                            return True

    return False


In [64]:
# Swordfish

def swordfish_rows(board, candidates):
    for digit in range(1, 10):
        row_positions = {}

        for row in range(9):
            cols = []
            for col in range(9):
                if board[row][col] == 0 and digit in candidates[row][col]:
                    cols.append(col)

            if 2 <= len(cols) <= 3:
                row_positions[row] = set(cols)

        rows = list(row_positions.keys())

        for row_combo in combinations(rows, 3):
            union_cols = set()
            for row in row_combo:
                union_cols.update(row_positions[row])

            if len(union_cols) == 3:
                for row in range(9):
                    if row not in row_combo:
                        for col in union_cols:
                            if board[row][col] == 0:
                                before = candidates[row][col].copy()
                                candidates[row][col].discard(digit)

                                if candidates[row][col] != before:
                                    print(
                                        f'Swordfish: removed {digit} from cell ({row+1}, {col+1}) '
                                        f'using rows {row_combo} and cols {sorted(union_cols)}'
                                    )
                                    return True
    return False

def swordfish_cols(board, candidates):
    for digit in range(1, 10):
        col_positions = {}

        for col in range(9):
            rows = []
            for row in range(9):
                if board[row][col] == 0 and digit in candidates[row][col]:
                    rows.append(row)

            if 2 <= len(rows) <= 3:
                col_positions[col] = set(rows)

        cols = list(col_positions.keys())

        for col_combo in combinations(cols, 3):
            union_rows = set()
            for col in col_combo:
                union_rows.update(col_positions[col])

            if len(union_rows) == 3:
                for col in range(9):
                    if col not in col_combo:
                        for row in union_rows:
                            if board[row][col] == 0:
                                before = candidates[row][col].copy()
                                candidates[row][col].discard(digit)

                                if candidates[row][col] != before:
                                    print(
                                        f'Swordfish: removed {digit} from cell ({row+1}, {col+1}) '
                                        f'using cols {col_combo} and rows {sorted(union_rows)}'
                                    )
                                    return True
    return False

def swordfish(board, candidates):
    if swordfish_rows(board, candidates):
        return True
    if swordfish_cols(board, candidates):
        return True
    return False

In [65]:
# Skyscraper

def skyscraper_rows(board, candidates):
    for digit in range(1, 10):
        row_positions = {}

        for row in range(9):
            cols = []
            for col in range(9):
                if board[row][col] == 0 and digit in candidates[row][col]:
                    cols.append(col)

            if len(cols) == 2:
                row_positions[row] = set(cols)

        rows = list(row_positions.keys())

        for i in range(len(rows)):
            for j in range(i + 1, len(rows)):
                r1, r2 = rows[i], rows[j]

                common_cols = row_positions[r1] & row_positions[r2]

                if len(common_cols) == 1:
                    shared_col = next(iter(common_cols))
                    roof1_col = next(iter(row_positions[r1] - {shared_col}))
                    roof2_col = next(iter(row_positions[r2] - {shared_col}))

                    roof1 = (r1, roof1_col)
                    roof2 = (r2, roof2_col)

                    common_peers = get_peers(*roof1) & get_peers(*roof2)

                    for row, col in common_peers:
                        if board[row][col] == 0:
                            before = candidates[row][col].copy()
                            candidates[row][col].discard(digit)

                            if candidates[row][col] != before:
                                print(
                                    f'Skyscraper: removed {digit} from ({row+1}, {col+1}) '
                                    f'using rows {r1+1},{r2+1} with shared col {shared_col+1} '
                                    f'and roofs {roof1}, {roof2}'
                                )
                                return True
    return False

def skyscraper_cols(board, candidates):
    for digit in range(1, 10):
        col_positions = {}

        for col in range(9):
            rows = []
            for row in range(9):
                if board[row][col] == 0 and digit in candidates[row][col]:
                    rows.append(row)

            if len(rows) == 2:
                col_positions[col] = set(rows)

        cols = list(col_positions.keys())

        for i in range(len(cols)):
            for j in range(i + 1, len(cols)):
                c1, c2 = cols[i], cols[j]

                common_rows = col_positions[c1] & col_positions[c2]

                if len(common_rows) == 1:
                    shared_row = next(iter(common_rows))
                    roof1_row = next(iter(col_positions[c1] - {shared_row}))
                    roof2_row = next(iter(col_positions[c2] - {shared_row}))

                    roof1 = (roof1_row, c1)
                    roof2 = (roof2_row, c2)

                    common_peers = get_peers(*roof1) & get_peers(*roof2)

                    for row, col in common_peers:
                        if board[row][col] == 0:
                            before = candidates[row][col].copy()
                            candidates[row][col].discard(digit)

                            if candidates[row][col] != before:
                                print(
                                    f'Skyscraper: removed {digit} from ({row+1}, {col+1}) '
                                    f'using cols {c1+1},{c2+1} with shared row {shared_row+1} '
                                    f'and roofs {roof1}, {roof2}'
                                )
                                return True
    return False

def skyscraper(board, candidates):
    if skyscraper_rows(board, candidates):
        return True
    if skyscraper_cols(board, candidates):
        return True
    return False

In [66]:
# Validation checks
def is_valid_row(board, row):
    desired_set = {1,2,3,4,5,6,7,8,9}
    digits = set()

    for cell in range(9):
        digits.add(board[row][cell])

    if digits == desired_set:
        return True
    else:
        return False

def is_valid_col(board, col):
    desired_set = {1,2,3,4,5,6,7,8,9}
    digits = set()
    for cell in range(9):
        digits.add(board[cell][col])

    if digits == desired_set:
        return True
    else:
        return False

def is_valid_block(board, row, col):
    desired_set = {1,2,3,4,5,6,7,8,9}
    digits = set()

    for r in range(row, row+3):
        for c in range(col, col+3):
            digits.add(board[r][c])

    if digits == desired_set:
        return True
    else:
        return False

def is_solved(board):
    errors = 0
    for row in range(9):
        is_valid = is_valid_row(board, row)
        if not is_valid:
            errors += 1
            print(f'Board not valid at row {row+1}')

    for col in range(9):
        is_valid = is_valid_col(board, col)
        if not is_valid:
            errors += 1
            print(f'Board not valid at col {col+1}')

    for row in range(0,9,3):
        for col in range(0,9,3):
            is_valid = is_valid_block(board, row, col)
            if not is_valid:
                errors += 1
                print(f'Board not valid at block ({row+1}, {col+1})')

    if errors == 0:
        print(f'Board is solved, great success')
    else:
        print(f'Board is obviously not solved :(')


In [67]:
def run_solver(board):
    candidates = get_all_candidates(board)
    while True:

        if naked_singles(board, candidates):
            candidates = get_all_candidates(board)
            continue

        if hidden_singles_rows(board, candidates):
            candidates = get_all_candidates(board)
            continue

        if hidden_singles_columns(board, candidates):
            candidates = get_all_candidates(board)
            continue

        if hidden_singles_boxes(board, candidates):
            candidates = get_all_candidates(board)
            continue

        if naked_pairs(board, candidates):
            continue

        if naked_triples(board, candidates):
            continue

        if hidden_pairs(board, candidates):
            continue

        if hidden_triples(board, candidates):
            continue

        if pointing_pairs_triples(board, candidates):
            continue

        if box_line_reduction(board, candidates):
            continue

        if x_wing(board, candidates):
            continue

        if y_wing(board, candidates):
            continue

        if swordfish(board, candidates):
            continue

        if skyscraper(board, candidates):
            continue

        print("No more progress can be made.")
        draw_board(board)
        break

run_solver(board)


Naked single: placed 9 at cell (2,7)
Hidden single (row): placed 3 placed at cell (2,3)
Hidden single (row): placed 6 placed at cell (2,9)
Hidden single (row): placed 6 placed at cell (3,3)
Hidden single (row): placed 4 placed at cell (8,7)
Hidden single (row): placed 4 placed at cell (7,2)
Hidden single (row): placed 1 placed at cell (8,3)
Hidden single (row): placed 1 placed at cell (9,8)
Hidden single (row): placed 1 placed at cell (1,7)
Naked single: placed 7 at cell (3,7)
Naked single: placed 2 at cell (3,2)
Naked single: placed 5 at cell (2,1)
Naked single: placed 5 at cell (3,8)
Naked single: placed 2 at cell (1,8)
Naked single: placed 4 at cell (2,6)
Naked single: placed 5 at cell (1,5)
Naked single: placed 2 at cell (2,5)
Hidden single (row): placed 4 placed at cell (5,5)
Hidden single (row): placed 6 placed at cell (5,6)
Hidden single (row): placed 1 placed at cell (5,4)
Naked single: placed 9 at cell (3,4)
Naked single: placed 1 at cell (3,6)
Hidden single (row): placed 2 pl

In [68]:
print()

In [69]:
event_log = [
    (1, 0, 2, 4, "naked_single", "placement"),
    (2, 4, 6, 7, "naked_pair", "elimination"),
    (3, 4, 7, 3, "hidden_single_row", "placement"),
    (4, 3, 3, 5, "x_wing", "elimination"),
    (5, 7, 1, 8, "hidden_single_col", "placement"),
    (6, 2, 5, 1, "pointing_pair", "elimination"),
    (7, 8, 8, 9, "naked_single", "placement"),
    (8, 5, 4, 2, "box_line_reduction", "elimination"),
    (9, 1, 0, 6, "hidden_single_box", "placement"),
    (10, 6, 2, 7, "y_wing", "elimination"),
    ]

In [70]:
import plotly.graph_objects as go

def plot_sample_plotly(event_log):
    xs = [col + 1 for step, row, col, digit, tactic, event_type in event_log]
    ys = [row + 1 for step, row, col, digit, tactic, event_type in event_log]
    zs = [step for step, row, col, digit, tactic, event_type in event_log]

    colors = ["red" if event_type == "placement" else "blue"
              for step, row, col, digit, tactic, event_type in event_log]

    texts = [
        f"step={step}<br>row={row+1}<br>col={col+1}<br>digit={digit}<br>tactic={tactic}<br>type={event_type}"
        for step, row, col, digit, tactic, event_type in event_log
    ]

    fig = go.Figure()

    fig.add_trace(go.Scatter3d(
        x=xs,
        y=ys,
        z=zs,
        mode='lines+markers+text',
        line=dict(color='gray', width=4),
        marker=dict(size=5, color=colors),
        text=[str(digit) for step, row, col, digit, tactic, event_type in event_log],
        hovertext=texts,
        hoverinfo='text'
    ))

    fig.update_layout(
        scene=dict(
            xaxis_title='Column',
            yaxis_title='Row',
            zaxis_title='Step'
        ),
        title='Sudoku Solve Path (3D)'
    )

    fig.show()

In [71]:
plot_sample_plotly(event_log)